# Camada Bronze — Decisões e premissas

## Armazenamento e estratégia de carga

- Os arquivos CSV recebidos são armazenados como Delta Tables no schema `bronze`, dentro do catálogo `v_credit_medallion`.
- Cada arquivo de origem possui uma tabela correspondente na camada Bronze.
- Quando uma carga é necessária, a tabela de destino é sobrescrita utilizando `OVERWRITE`, substituindo integralmente os dados da versão anterior.

### Justificativa

Não foi especificado se os arquivos recebidos representam cargas completas ou incrementais. Para este projeto, assumimos que cada arquivo representa um snapshot completo da respectiva fonte.

Essa premissa permite utilizar `OVERWRITE` sem a necessidade de identificar individualmente os registros inseridos, atualizados ou removidos entre versões.

Caso as fontes passem a fornecer atualizações incrementais ou parciais, a estratégia deverá ser reavaliada.

## Decisões baseadas na análise dos dados

A inspeção inicial dos arquivos identificou características que influenciam o desenho da ingestão e o tratamento posterior:

- **Chaves candidatas:** foram identificadas colunas únicas e sem valores nulos em `Chamados_Hora`, `Atendentes`, `Motivos`, `Canais`, `Clientes` e `Pesquisa_Satisfacao`.
- **Duplicatas exatas:** `Chamados` apresenta 2.636 linhas duplicadas, enquanto `Custos` apresenta 781.
- **Inconsistências:** foram observadas variações de formato, valores nulos e representações diferentes para informações equivalentes em algumas fontes.

Embora a existência de chaves candidatas possa auxiliar na identificação dos registros, ela não determina, por si só, a estratégia de atualização. Como não há uma definição sobre a semântica das cargas nem sobre o tratamento de registros removidos, não adotamos `MERGE` nesta etapa.

## Rastreabilidade

Todas as tabelas ingeridas recebem a coluna `_ingested_at`, preenchida com o timestamp da execução responsável pela carga.

Essa coluna permite identificar quando a versão atualmente armazenada foi ingerida.

Como a estratégia utiliza `OVERWRITE`, o timestamp não representa a data de criação ou de alteração individual de cada registro na fonte. Em uma nova ingestão, os valores dessa coluna são atualizados para os registros carregados.

## Idempotência

A pipeline foi projetada para evitar a duplicação de dados causada pela repetição de uma execução.

Como cada carga necessária substitui integralmente a tabela de destino, executar novamente a ingestão com a mesma entrada não acumula registros entre execuções.

Entretanto, `_ingested_at` é atualizado a cada nova carga. Portanto, a idempotência se aplica aos dados de negócio, e não à igualdade integral de todas as colunas da tabela.

## Preservação e qualidade dos dados

A camada Bronze não realiza transformações de negócio, como deduplicação, normalização ou correção de inconsistências. Dessa forma, preserva uma representação fiel dos dados recebidos e permite que as camadas Silver e Gold apliquem suas transformações de maneira explícita e rastreável.

Os arquivos são lidos com configurações específicas de codificação quando necessário, garantindo a interpretação correta dos caracteres presentes nas fontes.

## Detecção de alterações e otimização da ingestão

Para evitar leituras e escritas desnecessárias, a pipeline verifica os metadados dos arquivos antes de processá-los.

O processo funciona da seguinte forma:

1. Lista os arquivos disponíveis no diretório de landing e verifica a presença de todas as fontes esperadas.
2. Obtém o tamanho e a data de modificação de cada arquivo.
3. Compara esses valores com o estado registrado após a última ingestão bem-sucedida.
4. Se os metadados forem iguais e a tabela de destino existir, ignora o processamento daquele arquivo.
5. Se houver alteração nos metadados ou se a tabela de destino não existir, lê o CSV e sobrescreve a tabela correspondente.
6. Após a gravação bem-sucedida da tabela, atualiza o registro do arquivo na tabela de controle.

### Tabela de controle

O estado das ingestões é mantido em `bronze.tb_ingestion_control`, uma tabela Delta que registra, para cada arquivo:

- `file_name`: nome do arquivo de origem;
- `file_size`: tamanho do arquivo na última ingestão registrada;
- `modification_time`: data de modificação registrada;
- `target_table`: tabela Bronze correspondente;
- `last_ingested_at`: timestamp da última ingestão registrada como concluída.

O estado é atualizado somente após a gravação da tabela de destino. Assim, uma falha durante a leitura ou gravação não registra antecipadamente a ingestão como concluída.

### Limitações da estratégia

A comparação de tamanho e data de modificação é uma otimização baseada em metadados, não uma garantia de que o conteúdo permaneceu inalterado. Uma alteração que preserve ambos os valores pode não ser detectada.

Caso essa limitação se torne relevante, a estratégia poderá ser complementada ou substituída por uma comparação baseada em hash do conteúdo.

A implementação também pressupõe execuções sequenciais da pipeline. Execuções simultâneas exigiriam mecanismos adicionais de controle de concorrência.

# Configurações e Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
)

catalog = "v_credit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

landing_path = f"/Volumes/{catalog}/bronze/landing"
control_table = f"{bronze_schema}.tb_ingestion_control"

encodings = {
    "Chamados_Hora.CSV": "windows-1252",
    "base_atendentes.csv": "UTF-8",
    "base_motivos.csv": "UTF-8",
    "canais.csv": "UTF-8",
    "chamados.csv": "UTF-8",
    "clientes.csv": "windows-1252",
    "custos.csv": "UTF-8",
    "pesquisa_satisfacao.csv": "UTF-8",
}

# Arquivo de origem -> tabela Bronze e codificação
fontes = {
    "Chamados_Hora.CSV": {
        "tabela": "tb_chamados_hora",
        "encoding": encodings["Chamados_Hora.CSV"],
    },
    "base_atendentes.csv": {
        "tabela": "tb_atendentes",
        "encoding": encodings["base_atendentes.csv"],
    },
    "base_motivos.csv": {
        "tabela": "tb_motivos",
        "encoding": encodings["base_motivos.csv"],
    },
    "canais.csv": {
        "tabela": "tb_canais",
        "encoding": encodings["canais.csv"],
    },
    "chamados.csv": {
        "tabela": "tb_chamados",
        "encoding": encodings["chamados.csv"],
    },
    "clientes.csv": {
        "tabela": "tb_clientes",
        "encoding": encodings["clientes.csv"],
    },
    "custos.csv": {
        "tabela": "tb_custos",
        "encoding": encodings["custos.csv"],
    },
    "pesquisa_satisfacao.csv": {
        "tabela": "tb_pesquisa_satisfacao",
        "encoding": encodings["pesquisa_satisfacao.csv"],
    },
}

In [0]:
ambiente = dbutils.widgets.get("ambiente")
is_dev = ambiente == "dev"

# Cria estrutura medalhão

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("Estrutura criada com sucesso.")

# Inicializa a tabela de controle

In [0]:
control_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("file_size", LongType(), False),
    StructField("modification_time", LongType(), False),
    StructField("target_table", StringType(), False),
    StructField("last_ingested_at", TimestampType(), False),
])

# Cria a tabela apenas se ela ainda não existir
if not spark.catalog.tableExists(control_table):
    (
        spark.createDataFrame([], control_schema)
        .write.format("delta")
        .saveAsTable(control_table)
    )

print(f"Tabela de controle pronta: {control_table}")

# Função de Leitura

In [0]:
def ler_csv(path, encoding):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", encoding)
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", False)
        .csv(path)
    )

# Ingestao incremental por metadados

In [0]:
# Recupera o estado da última ingestão
controle = {
    row["file_name"]: row.asDict()
    for row in spark.table(control_table).collect()
}

# Lista os arquivos disponíveis na landing
arquivos_landing = {
    arquivo.name: arquivo
    for arquivo in dbutils.fs.ls(landing_path)
    if arquivo.name.lower().endswith(".csv")
}

# Verifica todas as fontes antes de iniciar a ingestão
faltantes = set(fontes) - set(arquivos_landing)

if faltantes:
    raise FileNotFoundError(
        f"Arquivos esperados ausentes na landing: {sorted(faltantes)}"
    )

for nome_arquivo, config in fontes.items():
    arquivo = arquivos_landing[nome_arquivo]
    nome_tabela = config["tabela"]
    tabela_completa = f"{bronze_schema}.{nome_tabela}"

    tamanho = int(arquivo.size)
    modificacao = int(arquivo.modificationTime)

    estado = controle.get(nome_arquivo)
    tabela_existe = spark.catalog.tableExists(tabela_completa)

    inalterado = (
        estado is not None
        and estado["file_size"] == tamanho
        and estado["modification_time"] == modificacao
        and estado["target_table"] == nome_tabela
        and tabela_existe
    )

    if inalterado:
        print(f"[SKIP] {nome_arquivo}")
        continue

    print(f"[LOAD] {nome_arquivo}")

    df = ler_csv(arquivo.path, config["encoding"])
    ingested_at = spark.sql(
        "SELECT current_timestamp() AS ts"
    ).first()["ts"]

    (
        df.withColumn(
            "_ingested_at",
            F.lit(ingested_at).cast("timestamp"),
        )
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_completa)
    )

    # Registra o estado após a gravação bem-sucedida
    controle[nome_arquivo] = {
        "file_name": nome_arquivo,
        "file_size": tamanho,
        "modification_time": modificacao,
        "target_table": nome_tabela,
        "last_ingested_at": ingested_at,
    }

    (
        spark.createDataFrame(
            list(controle.values()),
            schema=control_schema,
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(control_table)
    )

    print(f"[OK] {tabela_completa}")

print("Pipeline Bronze finalizada.")

# Validar as tabelas

In [0]:
if is_dev:
    for config in fontes.values():
        tabela = f"{bronze_schema}.{config['tabela']}"

        if spark.catalog.tableExists(tabela):
            df = spark.table(tabela)
            print(
                f"{tabela}: {df.count():,} registros | "
                f"{len(df.columns)} colunas"
            )

    print("\nTabela de controle:")
    spark.table(control_table).orderBy("file_name").show(truncate=False)
else:
    print(f"Validação final ignorada no ambiente {ambiente}.")